In [1]:
# Consolidated imports
import os
import urllib.request
import ssl
import certifi
import pandas as pd
import numpy as np

# SSL context for urllib requests using certifi CA bundle
ssl_context = ssl.create_default_context(cafile=certifi.where())

In [2]:
# Ensure output directory for monthly NYCCAS CSVs exists
out_dir = "data_csv/NYCCAS"
os.makedirs(out_dir, exist_ok=True)
print(f"Ensured directory exists: {out_dir}")

# iterate from Oct 2019 through Feb 2026 and save each month’s file
for year in range(2019, 2027):
    for month in range(1, 13):
        # stop when we have written February…2026
        if year == 2026 and month > 2:
            break
        # skip months before October 2019
        if year == 2019 and month < 10:
            continue

        url = f"https://raw.githubusercontent.com/nychealth/nyccas-data/refs/heads/main/hist/csv/{year}/{month}.csv"
        try:
            resp = urllib.request.urlopen(url, context=ssl_context)
            df = pd.read_csv(resp)                  # read month’s data
            df.to_csv(f"data_csv/NYCCAS/{year}_{month:02d}.csv", index=False)  # save to disk
            print(f"saved {year}-{month:02d}.csv")
        except Exception as exc:
            print(f"failed to fetch {year}-{month:02d}: {exc}")

Ensured directory exists: data_csv/NYCCAS
saved 2019-10.csv
saved 2019-11.csv
saved 2019-12.csv
saved 2020-01.csv
saved 2020-02.csv
saved 2020-03.csv
saved 2020-04.csv
saved 2020-05.csv
saved 2020-06.csv
saved 2020-07.csv
saved 2020-08.csv
saved 2020-09.csv
saved 2020-10.csv
saved 2020-11.csv
saved 2020-12.csv
saved 2021-01.csv
saved 2021-02.csv
saved 2021-03.csv
saved 2021-04.csv
saved 2021-05.csv
saved 2021-06.csv
saved 2021-07.csv
saved 2021-08.csv
saved 2021-09.csv
saved 2021-10.csv
saved 2021-11.csv
saved 2021-12.csv
saved 2022-01.csv
saved 2022-02.csv
saved 2022-03.csv
saved 2022-04.csv
saved 2022-05.csv
saved 2022-06.csv
saved 2022-07.csv
saved 2022-08.csv
saved 2022-09.csv
saved 2022-10.csv
saved 2022-11.csv
saved 2022-12.csv
saved 2023-01.csv
saved 2023-02.csv
saved 2023-03.csv
saved 2023-04.csv
saved 2023-05.csv
saved 2023-06.csv
saved 2023-07.csv
saved 2023-08.csv
saved 2023-09.csv
saved 2023-10.csv
saved 2023-11.csv
saved 2023-12.csv
saved 2024-01.csv
saved 2024-02.csv
save

In [3]:
out_path = "data_csv/NYCCAS/site-info.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

station_raw = "https://raw.githubusercontent.com/nychealth/nyccas-data/refs/heads/main/portal/station-new.csv"
ssl_ctx = ssl.create_default_context(cafile=certifi.where())
try:
    with urllib.request.urlopen(station_raw, context=ssl_ctx) as resp:
        data = resp.read()
    with open(out_path, "wb") as f:
        f.write(data)
    print(f"Saved station-new.csv → {out_path}")
except Exception as exc:
    print(f"Failed to fetch station-new.csv from raw URL: {exc}")

Saved station-new.csv → data_csv/NYCCAS/site-info.csv


In [4]:
base_dir = "data_csv/NYCCAS"          # adjust if the path is elsewhere
all_frames = []

for root, dirs, files in os.walk(base_dir):
	for fn in files:
		if not fn.endswith(".csv") or fn == "site-info.csv":
			continue
		path = os.path.join(root, fn)
		stem = os.path.splitext(fn)[0]    # e.g. "2020_01"
		parts = stem.split("_")
		if len(parts) < 2:
			# not a year_month file
			continue
		try:
			yr = int(parts[0])
			mo = int(parts[1])
		except ValueError:
			continue
		df_month = pd.read_csv(path)
		all_frames.append(df_month)

# merge all the monthly frames into one
merged = pd.concat(all_frames, ignore_index=True)

# save result
merged.to_csv("data_csv/NYCCAS_2019-2026.csv", index=False)

print("merged", len(all_frames), "files;", merged.shape, "rows in output")

merged 77 files; (374373, 4) rows in output


In [5]:
# read the site‐info file that lives alongside the monthly csvs
site_info = pd.read_csv(os.path.join(base_dir, "site-info.csv"))
print("columns:", site_info.columns.tolist())

# typically the file has a SiteID column and something like SiteName;
# adjust the name below if it is different in your copy
name_col = "Location" if "Location" in site_info.columns else site_info.columns[1]

# merge the name into the big frame and give it a consistent column name
merged = merged.merge(site_info[["SiteID", name_col]],
                      on="SiteID",
                      how="left")
merged = merged.rename(columns={name_col: "SiteName"})

# write the augmented table back out (overwrite or write a new file)
merged.to_csv("data_csv/NYCCAS_2019-2026.csv", index=False)

columns: ['SiteID', 'Location', 'loc_col', 'Latitude', 'Longitude']


In [6]:
# read the merged table from disk (the variable `merged` is already in the
# namespace, but read again to make sure we have the on‑disk copy)
df = pd.read_csv("data_csv/NYCCAS_2019-2026.csv")

# convert the timestamp column and extract month/year directly
# (ObservationTimeUTC has format yyyy-mm-dd hh:mm:ss...)
df["ObservationTimeUTC"] = pd.to_datetime(df["ObservationTimeUTC"])
df["month"] = df["ObservationTimeUTC"].dt.month
df["year"] = df["ObservationTimeUTC"].dt.year

In [7]:
# Example dataframe (replace with your actual df)
# df = pd.read_csv("your_file.csv")

# Mapping dictionary
site_to_borough = {
    'Midtown West': 'Manhattan',
    'Queensboro Bridge': 'Queens',
    'Williamsburg Bridge': 'Brooklyn',
    'Queens College': 'Queens',
    'Cross Bronx Expy': 'Bronx',
    'Manhattan Bridge': 'Manhattan',
    'Broadway/35th St': 'Manhattan',
    'Mott Haven': 'Bronx',
    'FDR': 'Manhattan',  # FDR Drive
    'Van Wyck': 'Queens',  # Van Wyck Expressway
    'SI Expwy': 'Staten Island',
    'BQE': 'Brooklyn',  # Brooklyn-Queens Expressway (choose dominant borough)
    'Hamilton Bridge': 'Bronx',
    "Hunt's Point": 'Bronx',
    'Glendale': 'Queens'
}

# Map boroughs
df['Borough'] = df['SiteName'].map(site_to_borough)

# Drop rows where Borough is NaN
df = df.dropna(subset=['Borough'])

In [8]:
winter = df[df["month"].isin([12, 1, 2])].copy()

# average the Value column, grouped by site (and by winter year)
winter_avg = (winter
              .groupby(["year"], as_index=False)["Value"]
              .mean())

winter_avg.to_csv("data_csv/NYCCAS-Winter-yearly.csv", index=False)
print("saved", winter_avg.shape[0], "rows to NYCCAS-Winter-yearly.csv")

saved 8 rows to NYCCAS-Winter-yearly.csv


In [9]:
# keep only July, August and September observations
summer = df[df["month"].isin([6, 7, 8])].copy()

# use the regular year as the summer identifier
summer["Year"] = summer["year"]

# average the Value column, grouped by site (and by summer year)
summer_avg = (summer
              .groupby(["Year"], as_index=False)["Value"]
              .mean())

summer_avg.to_csv("data_csv/NYCCAS-Summer-yearly.csv", index=False)
print("saved", summer_avg.shape[0], "rows to NYCCAS-Summer-yearly.csv")
# keep only July, August and September observations
spring = df[df["month"].isin([3, 4, 5])].copy()

# use the regular year as the spring identifier
spring["Year"] = spring["year"]

# average the Value column, grouped by site (and by spring year)
spring_avg = (spring
              .groupby(["Year"], as_index=False)["Value"]
              .mean())

spring_avg.to_csv("data_csv/NYCCAS-Spring-yearly.csv", index=False)
print("saved", spring_avg.shape[0], "rows to NYCCAS-Spring-yearly.csv")
# keep only July, August and September observations
fall = df[df["month"].isin([9, 10, 11])].copy()

# use the regular year as the fall identifier
fall["Year"] = fall["year"]

# average the Value column, grouped by site (and by fall year)
fall_avg = (fall
            .groupby(["Year"], as_index=False)["Value"]
            .mean())

fall_avg.to_csv("data_csv/NYCCAS-Fall-yearly.csv", index=False)
print("saved", fall_avg.shape[0], "rows to NYCCAS-Fall-yearly.csv")

saved 6 rows to NYCCAS-Summer-yearly.csv
saved 6 rows to NYCCAS-Spring-yearly.csv
saved 7 rows to NYCCAS-Fall-yearly.csv


In [10]:

# keep only December, January and February observations
winter = df[df["month"].isin([12, 1, 2])].copy()

# assign a winter‑year: December belongs to the following year’s winter
winter["Year"] = np.where(winter["month"] == 12,
                                 winter["year"] + 1,
                                 winter["year"])

# average the Value column, grouped by site (and by winter year)
winter_avg = (winter
              .groupby(["SiteName", "Year"], as_index=False)["Value"]
              .mean())

winter_avg.to_csv("data_csv/NYCCAS-Winter.csv", index=False)
print("saved", winter_avg.shape[0], "rows to NYCCAS-Winter.csv")

saved 57 rows to NYCCAS-Winter.csv


In [11]:
# keep only July, August and September observations
summer = df[df["month"].isin([6, 7, 8])].copy()

# use the regular year as the summer identifier
summer["Year"] = summer["year"]

# average the Value column, grouped by site (and by summer year)
summer_avg = (summer
              .groupby(["SiteName", "Year"], as_index=False)["Value"]
              .mean())

summer_avg.to_csv("data_csv/NYCCAS-Summer.csv", index=False)
print("saved", summer_avg.shape[0], "rows to NYCCAS-Summer.csv")

saved 49 rows to NYCCAS-Summer.csv


In [12]:
# keep only July, August and September observations
spring = df[df["month"].isin([3, 4, 5])].copy()

# use the regular year as the spring identifier
spring["Year"] = spring["year"]

# average the Value column, grouped by site (and by spring year)
spring_avg = (spring
              .groupby(["SiteName", "Year"], as_index=False)["Value"]
              .mean())

spring_avg.to_csv("data_csv/NYCCAS-Spring.csv", index=False)
print("saved", spring_avg.shape[0], "rows to NYCCAS-Spring.csv")

saved 52 rows to NYCCAS-Spring.csv


In [13]:
# keep only July, August and September observations
fall = df[df["month"].isin([9, 10, 11])].copy()

# use the regular year as the fall identifier
fall["Year"] = fall["year"]

# average the Value column, grouped by site (and by fall year)
fall_avg = (fall
            .groupby(["SiteName", "Year"], as_index=False)["Value"]
            .mean())

fall_avg.to_csv("data_csv/NYCCAS-Fall.csv", index=False)
print("saved", fall_avg.shape[0], "rows to NYCCAS-Fall.csv")

saved 51 rows to NYCCAS-Fall.csv


In [14]:
# use the regular year as the fall identifier
df["Year"] = df["year"]

# average the Value column, grouped by site (and by year)
year_avg = (df
            .groupby(["SiteName", "Year"], as_index=False)["Value"]
            .mean())

year_avg.to_csv("data_csv/NYCCAS-Annual.csv", index=False)
print("saved", year_avg.shape[0], "rows to NYCCAS-Annual.csv")

saved 77 rows to NYCCAS-Annual.csv


In [15]:
# keep only November, December and January observations
winter = df[df["month"].isin([11, 12, 1])].copy()

# assign a winter‑year: November and December belong to the following year’s winter, January to the current year
winter["Year"] = np.where(winter["month"].isin([11, 12]),
                                 winter["year"] + 1,
                                 winter["year"])

# average the Value column, grouped by Borough and winter year
winter_avg = (winter
              .groupby(["Borough", "Year"], as_index=False)["Value"]
              .mean())

winter_avg.to_csv("data_csv/NYCCAS-Winter-Borough.csv", index=False)
print("saved", winter_avg.shape[0], "rows to NYCCAS-Winter-Borough.csv")

saved 23 rows to NYCCAS-Winter-Borough.csv


In [16]:
# keep only June, July and August observations
summer = df[df["month"].isin([6, 7, 8])].copy()

# use the regular year as the summer identifier
summer["Year"] = summer["year"]

# average the Value column, grouped by Borough and summer year
summer_avg = (summer
			  .groupby(["Borough", "Year"], as_index=False)["Value"]
			  .mean())

summer_avg.to_csv("data_csv/NYCCAS-Summer-Borough.csv", index=False)
print("saved", summer_avg.shape[0], "rows to NYCCAS-Summer-Borough.csv")

# keep only September, October and November observations
fall = df[df["month"].isin([9, 10, 11])].copy()

# use the regular year as the fall identifier
fall["Year"] = fall["year"]

# average the Value column, grouped by Borough and fall year
fall_avg = (fall
			.groupby(["Borough", "Year"], as_index=False)["Value"]
			.mean())

fall_avg.to_csv("data_csv/NYCCAS-Fall-Borough.csv", index=False)
print("saved", fall_avg.shape[0], "rows to NYCCAS-Fall-Borough.csv")

# keep only March, April and May observations
spring = df[df["month"].isin([3, 4, 5])].copy()

# use the regular year as the spring identifier
spring["Year"] = spring["year"]

# average the Value column, grouped by Borough and spring year
spring_avg = (spring
			  .groupby(["Borough", "Year"], as_index=False)["Value"]
			  .mean())

spring_avg.to_csv("data_csv/NYCCAS-Spring-Borough.csv", index=False)
print("saved", spring_avg.shape[0], "rows to NYCCAS-Spring-Borough.csv")

saved 22 rows to NYCCAS-Summer-Borough.csv
saved 22 rows to NYCCAS-Fall-Borough.csv
saved 24 rows to NYCCAS-Spring-Borough.csv


In [17]:
# Add season column to each seasonal average DataFrame
winter_avg['Season'] = 'Winter'
summer_avg['Season'] = 'Summer'
spring_avg['Season'] = 'Spring'
fall_avg['Season'] = 'Fall'

# Combine all seasonal averages into one DataFrame
combined_seasonal = pd.concat([winter_avg, summer_avg, spring_avg, fall_avg], ignore_index=True)

# Save to CSV
combined_seasonal.to_csv("data_csv/NYCCAS-Seasonal-Borough.csv", index=False)
print("saved", combined_seasonal.shape[0], "rows to NYCCAS-Seasonal-Borough.csv")

saved 91 rows to NYCCAS-Seasonal-Borough.csv


In [18]:
# Assuming df has columns: "year", "month", "Borough", "Value"

# Group by Borough and Year, and calculate the yearly average of Value
yearly_avg = df.groupby(["Borough", "year"], as_index=False)["Value"].mean()

# Save to CSV
yearly_avg.to_csv("data_csv/NYCCAS-Yearly-Borough.csv", index=False)

print("saved", yearly_avg.shape[0], "rows to NYCCAS-Yearly-Borough.csv")

saved 33 rows to NYCCAS-Yearly-Borough.csv
